# FASTQ to h5ad Pipeline (kallisto | bustools)
This pipeline aligns raw single-cell RNA-seq FASTQ files to the mm10 mouse reference genome and generates `.h5ad` count matrices.

### Steps:
1. **Reference Generation:** Build the transcriptome index from the genome `.fa` and `.gtf`.
2. **Quantification:** Align reads and count UMIs to generate cell x gene matrices.


In [2]:
import os
import glob

# 1. Define Base Directories
base_dir = '/home/nakagawa/datasets'
fastq_dir = os.path.join(base_dir, 'SRR_cortex/SRR_cortex_scRNA')
genome_dir = os.path.join(base_dir, 'genome', 'mm10')
output_dir = os.path.join(base_dir, 'SRR_cortex/SRR_cortex_scRNA/processed_h5ad')

os.makedirs(output_dir, exist_ok=True)

# 2. Define Reference Files
fasta_path = os.path.join(genome_dir, 'Mus_musculus.GRCm38.dna.primary_assembly.fa')
gtf_path = os.path.join(genome_dir, 'Mus_musculus.GRCm38.84.gtf')

# 3. Define Index Output Paths
index_path = os.path.join(genome_dir, 'transcriptome.idx')
t2g_path = os.path.join(genome_dir, 'transcripts_to_genes.txt')
t_fasta_path = os.path.join(genome_dir, 'transcriptome.fa')

print("Directories and paths configured.")


Directories and paths configured.


In [ ]:
%%bash

FASTQ_DIR="/home/nakagawa/datasets/SRR_cortex/SRR_cortex_scRNA"
OUT_DIR="/home/nakagawa/datasets/SRR_cortex/SRR_cortex_scRNA/processed_h5ad"

INDEX="/home/nakagawa/datasets/genome/mm10/transcriptome.idx"
T2G="/home/nakagawa/datasets/genome/mm10/transcripts_to_genes.txt"

mkdir -p "$OUT_DIR"

for R2 in ${FASTQ_DIR}/*_2.fastq.gz; do

    SAMPLE=$(basename "$R2" _2.fastq.gz)

    echo "---------------------------------------------------"
    echo "Processing $SAMPLE..."

    SAMPLE_OUT="$OUT_DIR/$SAMPLE"

    R1="$FASTQ_DIR/${SAMPLE}_1.fastq.gz"

    # Skip if R1 missing
    if [[ ! -f "$R1" ]]; then
        echo "Missing R1 for $SAMPLE"
        continue
    fi

    # Remove old failed outputs if they exist
    rm -rf "$SAMPLE_OUT"

    echo "Trying 10xv3..."

    kb count \
        -i "$INDEX" \
        -g "$T2G" \
        -x 10xv3 \
        -o "$SAMPLE_OUT" \
        --h5ad \
        "$R1" "$R2"

    STATUS=$?

    # If 10xv3 failed, retry with 10xv2
    if [[ $STATUS -ne 0 ]]; then

        echo "10xv3 failed for $SAMPLE"
        echo "Retrying with 10xv2..."

        rm -rf "$SAMPLE_OUT"

        kb count \
            -i "$INDEX" \
            -g "$T2G" \
            -x 10xv2 \
            -o "$SAMPLE_OUT" \
            --h5ad \
            "$R1" "$R2"

        STATUS=$?
    fi

    if [[ $STATUS -eq 0 ]]; then
        echo "Finished $SAMPLE successfully"
    else
        echo "FAILED: $SAMPLE"
    fi

done

---------------------------------------------------
Processing SRR12082755...
Trying 10xv3...


[2026-05-21 21:12:33,517]    INFO [count] Using index /home/nakagawa/datasets/genome/mm10/transcriptome.idx to generate BUS file to /home/nakagawa/datasets/SRR_cortex/SRR_cortex_scRNA/processed_h5ad/SRR12082755 from
[2026-05-21 21:12:33,517]    INFO [count]         /home/nakagawa/datasets/SRR_cortex/SRR_cortex_scRNA/SRR12082755_1.fastq.gz
[2026-05-21 21:12:33,517]    INFO [count]         /home/nakagawa/datasets/SRR_cortex/SRR_cortex_scRNA/SRR12082755_2.fastq.gz
[2026-05-21 21:19:04,832]    INFO [count] Sorting BUS file /home/nakagawa/datasets/SRR_cortex/SRR_cortex_scRNA/processed_h5ad/SRR12082755/output.bus to /home/nakagawa/datasets/SRR_cortex/SRR_cortex_scRNA/processed_h5ad/SRR12082755/tmp/output.s.bus
[2026-05-21 21:19:22,457]    INFO [count] On-list not provided
[2026-05-21 21:19:22,457]    INFO [count] Copying pre-packaged 10XV3 on-list to /home/nakagawa/datasets/SRR_cortex/SRR_cortex_scRNA/processed_h5ad/SRR12082755
[2026-05-21 21:19:23,033]    INFO [count] Inspecting BUS file /h

Finished SRR12082755 successfully
---------------------------------------------------
Processing SRR12082756...
Trying 10xv3...


[2026-05-21 21:20:05,023]    INFO [count] Using index /home/nakagawa/datasets/genome/mm10/transcriptome.idx to generate BUS file to /home/nakagawa/datasets/SRR_cortex/SRR_cortex_scRNA/processed_h5ad/SRR12082756 from
[2026-05-21 21:20:05,023]    INFO [count]         /home/nakagawa/datasets/SRR_cortex/SRR_cortex_scRNA/SRR12082756_1.fastq.gz
[2026-05-21 21:20:05,023]    INFO [count]         /home/nakagawa/datasets/SRR_cortex/SRR_cortex_scRNA/SRR12082756_2.fastq.gz


In [ ]:
# Identify unique sample prefixes (e.g., 'SRR12082755')
# Note: scRNA-seq typically requires both R1 (Barcode/UMI) and R2 (cDNA). 
# We assume standard pairing (_1.fastq.gz and _2.fastq.gz).

fastq_files = glob.glob(os.path.join(fastq_dir, '*_2.fastq.gz'))
sample_ids = sorted([os.path.basename(f).split('_2.fastq.gz')[0] for f in fastq_files])

print(f"Found {len(sample_ids)} samples to process: {sample_ids}")

# The specific scRNA-seq technology (e.g., 10x v2, 10x v3). 
# Adjust '10x_v3' below if your SRR data was sequenced using a different chemistry.
technology = '10xv3'


In [ ]:
%%bash -s "$fastq_dir" "$output_dir" "$index_path" "$t2g_path" "$technology" "$(echo ${sample_ids[@]})"
FASTQ_DIR=$1
OUT_DIR=$2
INDEX=$3
T2G=$4
TECH=$5
read -a SAMPLES <<< "$6"

for SAMPLE in "${SAMPLES[@]}"; do
    echo "---------------------------------------------------"
    echo "Processing $SAMPLE..."
    
    SAMPLE_OUT="$OUT_DIR/$SAMPLE"
    R1="$FASTQ_DIR/${SAMPLE}_1.fastq.gz"
    R2="$FASTQ_DIR/${SAMPLE}_2.fastq.gz"
    
    # kb count runs pseudoalignment and generates Anndata (.h5ad) directly
    kb count \
        -i $INDEX \
        -g $T2G \
        -x $TECH \
        -o $SAMPLE_OUT \
        --h5ad \
        --tcc \
        $R1 $R2
        
    echo "Finished $SAMPLE. Output saved to $SAMPLE_OUT"
done


In [ ]:
import scanpy as sc

# After Kallisto finishes, verify the generated .h5ad files
for sample in sample_ids:
    h5ad_path = os.path.join(output_dir, sample, 'counts_unfiltered', 'adata.h5ad')
    
    if os.path.exists(h5ad_path):
        adata = sc.read_h5ad(h5ad_path)
        print(f"{sample}: {adata.n_obs} cells x {adata.n_vars} genes")
        
        # Optional: Rename and move the file to the main processed directory
        # for easier access in your Backward Trajectory notebook
        os.rename(h5ad_path, os.path.join(output_dir, f"{sample}_raw.h5ad"))
    else:
        print(f"Warning: Processing failed or output missing for {sample}")
